# Heat Demand analysis and prediction Lag features for electricity demand Tuning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor

from scripts.notebook_utils import (
    evaluate_regression,
    mape,
    select_correlated_features,
    split_dataframe_chronologically,
    time_series_cv_evaluation,
)

np.random.seed(42)


In [ ]:
df = pd.read_csv("./data/Load_data_01.csv")
df["Time"] = pd.to_datetime(df["Time"])
df.set_index("Time", inplace=True)
df.describe().T

In [ ]:
df_heat_daily = pd.DataFrame()  # create a new dataframe for daily data

df_heat_daily["heat_demand_values"] = (  # sum up all heat_demand_values
    df["heat_demand_values"].resample("D").sum()
)

In [ ]:
# create a new dataframe for hourly data
df_heat_hourly = df["heat_demand_values"]
df_heat_hourly = pd.DataFrame(df_heat_hourly)

In [ ]:
lag_hours = np.arange(1,96, 1)  # create a list of lag hours

for lag in lag_hours:
    df_heat_hourly[f"t-{lag}"] = df_heat_hourly["heat_demand_values"].shift(lag)

df_heat_hourly.dropna(inplace=True)

In [ ]:
def corr_features_plot_h(df_heat_hourly, threshold):
    correlations = df_heat_hourly.corr()["heat_demand_values"]
    correlated_features = correlations[abs(correlations) > threshold].index
    _, ax = plt.subplots(figsize=(16, 16))
    sns.heatmap(
        df_heat_hourly.corr().loc[correlated_features, correlated_features],
        annot=True,
        ax=ax,
    )
    ax.set_title(f"Correlation Matrix of Daily Data (abs(corr) >= {threshold})")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_features_plot_h(df_heat_hourly, 0.8)

In [ ]:
correlations = df_heat_hourly.corr()["heat_demand_values"]
correlated_features = correlations[abs(correlations) > 0.8].index

In [ ]:
trainval_df, test_df = split_dataframe_chronologically(df_heat_hourly, test_size=0.3)
correlations, correlated_features = select_correlated_features(
    trainval_df, "heat_demand_values", 0.8
)

X_trainval = trainval_df[correlated_features].drop("heat_demand_values", axis=1)
y_trainval = trainval_df["heat_demand_values"]
X_test = test_df[correlated_features].drop("heat_demand_values", axis=1)
y_test = test_df["heat_demand_values"]


In [ ]:
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

In [ ]:
def MAPE(y_true, y_pred):  # Calculate Mean Absolute Percentage Error
    return mape(y_true, y_pred)


In [ ]:
print("AdaBoost Regressor with Decision Tree Regressor 5 Fold TimeSeriesSplit Results:\n")
fold_metrics_df, summary = time_series_cv_evaluation(
    ada_reg, X_trainval, y_trainval, n_splits=5
)
evaluation_df = pd.DataFrame([summary])
fold_metrics_df


In [ ]:
evaluation_df

In [ ]:
y_pred = ada_reg.predict(X_test)

In [ ]:
df_results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})

In [ ]:
rmse, mae, mape_score, r2 = evaluate_regression(y_test, y_pred)
print(
    f"Training on holdout set: RMSE: {rmse:.4f}, MAE: {mae:.4f}, "
    f"MAPE:{mape_score:.4f} %, R2: {r2:.4f}"
)
df_results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})


In [ ]:
len(df_results)

In [ ]:
df_results.plot(figsize=(16, 6))

In [ ]:
test_idx = df_results.index.sort_values()[300:1000]

In [ ]:
df_results.loc[test_idx].plot(figsize=(16, 6))

In [ ]:
df_air = df["air_temperature"]
df_air = pd.DataFrame(df_air)

In [ ]:
lag_hours = np.arange(1, 61, 1)

for lag in lag_hours:
    df_air[f"t-{lag}"] = df_air["air_temperature"].shift(lag)

df_air.dropna(inplace=True)

In [ ]:
def corr_features_plot_a(df_air, threshold):
    correlations = df_air.corr()["air_temperature"]
    correlated_features = correlations[abs(correlations) > threshold].index
    _, ax = plt.subplots(figsize=(16, 16))
    sns.heatmap(
        df_air.corr().loc[correlated_features, correlated_features],
        annot=True,
        ax=ax,
    )
    ax.set_title(f"Correlation Matrix of Daily Data (abs(corr) >= {threshold})")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_features_plot_a(df_air, 0.3)

In [ ]:
correlations = df_air.corr()["air_temperature"]
correlated_features = correlations[abs(correlations) > 0.5].index

In [ ]:
trainval_df, test_df = split_dataframe_chronologically(df_air, test_size=0.3)
correlations, correlated_features = select_correlated_features(
    trainval_df, "air_temperature", 0.5
)

X_trainval = trainval_df[correlated_features].drop("air_temperature", axis=1)
y_trainval = trainval_df["air_temperature"]
X_test = test_df[correlated_features].drop("air_temperature", axis=1)
y_test = test_df["air_temperature"]


In [ ]:
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

In [ ]:
def MAPE(y_true, y_pred):  # Calculate Mean Absolute Percentage Error
    return mape(y_true, y_pred)


In [ ]:
print("AdaBoost Regressor with Decision Tree Regressor 5 Fold TimeSeriesSplit Results:\n")
fold_metrics_df, summary = time_series_cv_evaluation(
    ada_reg, X_trainval, y_trainval, n_splits=5
)
evaluation_df = pd.DataFrame([summary])
fold_metrics_df


In [ ]:
evaluation_df

In [ ]:
y_pred = ada_reg.predict(X_test)

In [ ]:
df_results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})

In [ ]:
rmse, mae, mape_score, r2 = evaluate_regression(y_test, y_pred)
print(
    f"Training on holdout set: RMSE: {rmse:.4f}, MAE: {mae:.4f}, "
    f"MAPE:{mape_score:.4f} %, R2: {r2:.4f}"
)
df_results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})


In [ ]:
len(df_results)

In [ ]:
df_results.plot(figsize=(16, 6))

In [ ]:
test_idx = df_results.index.sort_values()[300:310]

In [ ]:
df_results.loc[test_idx].plot(figsize=(16, 6))

In [ ]:
lag_days = np.arange(1, 61, 1)

for lag in lag_days:
    df_heat_daily[f"heat_demand_lag_{lag}"] = df_heat_daily["heat_demand_values"].shift(
        lag
    )

df_heat_daily.dropna(inplace=True)

In [ ]:
df_heat_daily

In [ ]:
# Deprecated random-split experiment removed. The daily heat workflow below uses
# chronological train/test splits instead.


In [ ]:
def corr_features_plot(df_daily, threshold):
    correlations = df_daily.corr()["heat_demand_values"]
    correlated_features = correlations[abs(correlations) > threshold].index
    _, ax = plt.subplots(figsize=(16, 16))
    sns.heatmap(
        df_daily.corr().loc[correlated_features, correlated_features], annot=True, ax=ax
    )
    ax.set_title(f"Correlation Matrix of Daily Data (abs(corr) >= {threshold})")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_features_plot(df_heat_daily, 0.8)

In [ ]:
corr_features_plot(df_heat_daily, 0.5)

In [ ]:
corr_features_plot(df_heat_daily, 0.3)

In [ ]:
# corr_features_plot(df_heat_daily, 0.1)

In [ ]:
def train_test_set(df, start, end, split_time):
    train = df[(df.index > start) & (df.index <= split_time)]
    test = df[(df.index > split_time) & (df.index <= end)]
    X_train, y_train = (
        train.drop(["heat_demand_values"], axis=1),
        train["heat_demand_values"],
    )
    X_test, y_test = (
        test.drop(["heat_demand_values"], axis=1),
        test["heat_demand_values"],
    )
    return X_train, y_train, X_test, y_test

In [ ]:
def train_model(X_train, y_train, X_test, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = np.mean(np.abs(y_pred - y_test))
    r2 = r2_score(y_test, y_pred)
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R2: {r2:.4f}")
    df_results = pd.DataFrame(
        {"Time": y_test.index, "y_test": y_test, "y_pred": y_pred}
    )
    return df_results

In [ ]:
def plot_results(df_results, model_name):
    _, ax = plt.subplots(figsize=(12, 3))
    ax.plot(df_results["Time"], df_results["y_test"], label="Test")
    ax.plot(df_results["Time"], df_results["y_pred"], label="Predicted")
    ax.set_title(model_name)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_heat_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
dt_reg = DecisionTreeRegressor(max_depth=10, random_state=42)
df_results = train_model(X_train, y_train, X_test, y_test, dt_reg)

In [ ]:
plot_results(df_results, "Decision Tree Regressor")

In [ ]:
RF_reg = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)

df_results = train_model(X_train, y_train, X_test, y_test, RF_reg)

In [ ]:
plot_results(df_results, "Random Forest Regressor")

In [ ]:
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

df_results = train_model(X_train, y_train, X_test, y_test, ada_reg)

In [ ]:
plot_results(df_results, "AdaBoost Regressor with Decision Tree Regressor")

In [ ]:
features = df_heat_daily.columns.drop("heat_demand_values")

trainval_df, test_df = split_dataframe_chronologically(df_heat_daily, test_size=0.2)
X_trainval = trainval_df[features]
y_trainval = trainval_df["heat_demand_values"]
X_test = test_df[features]
y_test = test_df["heat_demand_values"]


In [ ]:
print("AdaBoost Regressor with Decision Tree Regressor 5 Fold TimeSeriesSplit Results:\n")
fold_metrics_df, summary = time_series_cv_evaluation(
    ada_reg, X_trainval, y_trainval, n_splits=5
)
evaluation_df = pd.DataFrame([summary])
fold_metrics_df


In [ ]:
evaluation_df


In [ ]:
fold_metrics_df


In [ ]:
y_pred = ada_reg.predict(X_test)

rmse, mae, _, r2 = evaluate_regression(y_test, y_pred)
print(f"Training on holdout set: RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

df_results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})


In [ ]:
df_results.plot(figsize=(16, 6))